# Quantitative Evaluation for `mailcom`

## Install packages if needed

Recommended: create a clean environment and and install necessary packages.

In [ ]:
# install Hugging Face datasets if needed
%pip install datasets

In [ ]:
# install mailcom/refine_email_checking if needed
%pip install git+https://github.com/ssciwr/mailcom.git@refine_email_checking

## Import necessary packages

In [ ]:
from datasets import load_dataset
import json

import mailcom
import pandas as pd
import ast
import re

## Email address detection

### Prepare data

We used Hugging Face `Josephgflowers/PII-NER` dataset for this evaluation.

Since we only focus on email address detection, we first filtered the dataset to obtain a subset of sentences that contain email addresses. We then applied the `mailcom` transformation to these sentences and compared the detected email addresses with the ground-truth annotations provided in the dataset.

In [ ]:
# load the dataset
pii_ner_ds = load_dataset("Josephgflowers/PII-NER")

In [ ]:
pii_ner_ds

In [ ]:
# create a copy of the dataset that contains only the original text and the extracted email addresses

def extract_text_emails(row: str):
    try:
        assistant_output = json.loads(row["assistant"])
        text = row["user"]
        emails = assistant_output.get("EMAIL", [])
    except:
        text = row["user"]
        emails = []
    return {"text": text, "emails": emails}

email_ds = pii_ner_ds["train"].map(extract_text_emails, remove_columns=pii_ner_ds["train"].column_names)
email_ds = email_ds.filter(lambda x: (len(x["emails"]) > 0) and "@" in x["text"])
email_ds # 2216 items

In [ ]:
# no need to run this if using mailcom/refine_email_checking branch
# email in this dataset usually followed by a comma
# for simplicity, we added spaces before and after the email addresses to separate email addresses from punctuations
def add_spaces_around_emails(text: str, emails: list):
    for email in emails:
        text = text.replace(email, f" {email} ")
    return text

email_ds = email_ds.map(lambda x: {"text": add_spaces_around_emails(x["text"], x["emails"])})

In [ ]:
# Delete bad items from the dataset
# 3 items where email addresses have ’ (non-standard ASCII apostrophe, 8217) instead of ' (standard ASCII apostrophe, 39)
# 14 items where email addresses have space in the middle
# 1 item where the email address is incorrect, 
# -- text starts with "As a resident of 707 Kade", 
# -- email "nishithchawla@bhavsar-buch.com" (the correct email is "nsithchawla@bhavsar-buch.com")
def is_valid_email(email: str, text: str):
    invalid_emails = (" " in email) or\
          (any(ord(c) == 8217 for c in email)) or\
              (email == "nishithchawla@bhavsar-buch.com" and text.startswith("As a resident of 707 Kade"))
    if invalid_emails:
        return False
    return True
def is_valid_item(emails: list, text: str):
    for email in emails:
        if not is_valid_email(email, text):
            return False
    return True

email_ds = email_ds.filter(lambda x: is_valid_item(x["emails"], x["text"]))
assert len(email_ds) == 2198, f"Expected 2198 items after filtering, but got {len(email_ds)}"
email_ds # 2198 items

In [ ]:
# save the filtered dataset to a csv file for evaluation
email_ds_df = pd.DataFrame(email_ds)
email_ds_df["emails"] = email_ds_df["emails"].apply(json.dumps)
email_ds_df.to_csv("eval/email_detection_eval.csv", index=False)

### Run `mailcom` on the evaluation dataset

In [ ]:
# load workflow configuration
new_settings = {
    "default_lang": "en", # only English in the dataset
    "pseudo_fields": ["content"], # we don't consider subject here
    "pseudo_emailaddresses": True, # we only pseudo email addresses
    "pseudo_ne": False,
    "pseudo_numbers": False,
    "datetime_detection": False,
}

# save the updated configuration to a file for reproducibility purposes
new_settings_dir = "./eval"
workflow_settings = mailcom.get_workflow_settings(new_settings=new_settings, 
                                                  updated_setting_dir= new_settings_dir,
                                                  save_updated_settings=True)

In [ ]:
# load csv file into input handler
input_csv = "eval/email_detection_eval.csv"
# the columns of the csv that should be passed through the processing pipeline/retained in the pipeline
matching_columns = ["text"]
# the predefined keys that should be used to match these columns, in the correct order
pre_defined_keys = ["content"]
# what to call any columns that are not matched to pre-defined keys
# get this from the workflow settings
unmatched_keyword = workflow_settings.get("csv_col_unmatched_keyword")

input_handler = mailcom.get_input_handler(in_path=input_csv, in_type="csv", 
                                          col_names=matching_columns, 
                                          init_data_fields=pre_defined_keys, 
                                          unmatched_keyword=unmatched_keyword)

In [ ]:
# process the input data
mailcom.process_data(input_handler.get_email_list(), workflow_settings) # 22.3s on laptop

In [ ]:
# convert the processed data into a dataframe
email_df = pd.DataFrame(input_handler.get_email_list())

In [ ]:
# only keep the content,  pseudo_content, and sentences columns
filtered_email_df = email_df[["content", "pseudo_content", "sentences"]]

In [ ]:
# add pseudo_content to the original dataset
org_email_df = pd.read_csv(input_csv)
# convert the emails column from string to list
org_email_df["emails"] = org_email_df["emails"].apply(ast.literal_eval)

# check before merging
(org_email_df["text"].sort_values().reset_index(drop=True) ==
 filtered_email_df["content"].sort_values().reset_index(drop=True)).all()

In [ ]:
# merge the pseudo_content into the original datasetn_df[
merged_email_df = org_email_df.merge(filtered_email_df, left_on="text", right_on="content", how="inner")

assert len(merged_email_df) == len(org_email_df) == len(filtered_email_df)

In [ ]:
merged_email_df = merged_email_df.drop(columns=["content"])
merged_email_df.head()

In [ ]:
prev_email_checking = False # set to True if using mailcom/main branch and refine_email_checking has not been merged yet

# create expected content column as ground truth for evaluation
def clean_sentence(sentence, prev_email_checking=True):
    if not prev_email_checking:
        # no need to clean the sentence if using regex for email checking
        return sentence
    # normalize whitespace to make it match the way mailcom clean the sentences
    normalized_sentence = " ".join(re.split(r"\s+", sentence))
    return normalized_sentence
    
def replace_email_with_placeholder(sentences, email_list):
    replaced_sents = []
    for sentence in sentences:
        normalized_sentence = clean_sentence(sentence, prev_email_checking)
        for email in email_list:
            replaced_sent = normalized_sentence.replace(email, "[email]")
            replaced_sents.append(replaced_sent)
    return " ".join(replaced_sents)

merged_email_df["expected_content"] = merged_email_df.apply(lambda row: replace_email_with_placeholder(row["sentences"].get("content"), row["emails"]), axis=1)

In [ ]:
merged_email_df.head()

In [ ]:
# calculate the exact match accuracy between the pseudo_content and the expected_content
merged_email_df["exact_match"] = merged_email_df.apply(lambda row: row["pseudo_content"] == row["expected_content"], axis=1)

In [ ]:
# calculate tp, fp, fn
def calculate_tp_fp_fn(row):
    pred_count = len(re.findall(r"\[email\]", row["pseudo_content"]))
    gold_count = len(re.findall(r"\[email\]", row["expected_content"]))

    true_positives = min(pred_count, gold_count)
    false_positives = max(pred_count - gold_count, 0)
    false_negatives = max(gold_count - pred_count, 0)
    
    # precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    # recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    # f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return pd.Series({"tp": true_positives, "fp": false_positives, "fn": false_negatives})

metrics_df = merged_email_df.apply(calculate_tp_fp_fn, axis=1)
merged_email_df = pd.concat([merged_email_df, metrics_df], axis=1)

In [ ]:
merged_email_df.head()

In [ ]:
# save the results to a csv file for further analysis
merged_email_df.to_csv("eval/email_detection_results.csv", index=False)

In [ ]:
def cal_eval_metrics(df):
    total_tp = df["tp"].sum()
    total_fp = df["fp"].sum()
    total_fn = df["fn"].sum()

    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    accuracy = df["exact_match"].mean()
    print(f"Exact match accuracy: {accuracy:.4f}")
    print(f"Micro Precision: {precision:.4f}")
    print(f"Micro Recall: {recall:.4f}")
    print(f"Micro F1 Score: {f1:.4f}")


In [ ]:
cal_eval_metrics(merged_email_df)
# Results when using simple check "@" in sentence
# Exact match accuracy: 0.7302
# Micro Precision: 0.7741
# Micro Recall: 1.0000
# Micro F1 Score: 0.8727
# recall > precision, as mailcom also marks some mentioning form, e.g. @username as email addresses.

# Results when using regex for email checking
# Exact match accuracy: 0.9995
# Micro Precision: 1.0000
# Micro Recall: 1.0000
# Micro F1 Score: 1.0000
# There is only one failed case:
# -- sentence: With an email address that reflects her professional prowess-bhamini.gulati@shukla.biz-she navigates the digital realm with ease.
# -- expected content: With an email address that reflects her professional prowess-[email]-she navigates the digital realm with ease.
# -- pseudo content: With an email address that reflects her professional [email]-she navigates the digital realm with ease.

## NER detection

### Prepare data

We use Hugging Face `Babelscape/wikineural` dataset (`test_en`, `11,597` rows) for this evaluation.

The dataset contains tokens and their corresponding NER tags.

`{'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ORG': 3, 'I-ORG': 4, 'B-LOC': 5, 'I-LOC': 6, 'B-MISC': 7, 'I-MISC': 8}`

We first converted the tokens into sentence and extract the named entities based on the NER tags. We then applied the `mailcom` transformation to these sentences and compared the detected named entities with the ground-truth annotations provided in the dataset.

In [ ]:
# load the dataset
ner_ds = load_dataset("Babelscape/wikineural", split="test_en")
ner_ds # 11597 items

In [ ]:
# NER tags
ner_tags_def = {'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ORG': 3, 'I-ORG': 4, 'B-LOC': 5, 'I-LOC': 6, 'B-MISC': 7, 'I-MISC': 8}
# convert key-value to value-key for easier mapping
ner_tags = {v: k for k, v in ner_tags_def.items()}
ner_tags

In [ ]:
# map int in ner_tags to their corresponding string labels
def map_ner_tags(row):
    ner_tags_int = row["ner_tags"]
    ner_tags_str = [ner_tags.get(tag, "O") for tag in ner_tags_int]
    return {"ner_tags_str": ner_tags_str}

ner_ds_mapped = ner_ds.map(map_ner_tags)
ner_ds_mapped

In [ ]:
# regex to clean up spaces surrounding non-word chars
space_before_punc = re.compile(r"\s+([?.!,;:%])")
multiple_spaces = re.compile(r"\s+")
space_after_open_bracket = re.compile(r"([(\[{])\s+")
space_before_close_bracket = re.compile(r"\s+([)\]}])")

apostrophe_between_words = re.compile(r"(\w)\s+[']\s+(\w)")
apostrophe_between_words_special = re.compile(r"(\w)\s+[’]\s+(\w)")

space_in_quotes = re.compile(r'(["“‘])\s+(.*?)\s+(["”’])') # assuming that single quotes are used as apostrophes, not quotation marks

# for each row, convert tokens into sentence or phrase and extract NE
def convert_tokens_to_phrase(tokens):
    sentence = " ".join(tokens)

    # post processing
    # remove space before punctuations
    sentence = space_before_punc.sub(r"\1", sentence)
    # replace multiple spaces with a single space
    sentence = multiple_spaces.sub(" ", sentence).strip()
    # fix space after opening brackets
    sentence = space_after_open_bracket.sub(r"\1", sentence)
    # fix space before closing brackets
    sentence = space_before_close_bracket.sub(r"\1", sentence)
    # fix apostrophes between words
    sentence = apostrophe_between_words.sub(r"\1'\2", sentence)
    sentence = apostrophe_between_words_special.sub(r"\1’\2", sentence)
    # fix space in quotes
    sentence = space_in_quotes.sub(r'\1\2\3', sentence)
    return sentence

def get_token_offsets(sentence: str, tokens: list):
    offsets = []
    current_pos = 0
    for token in tokens:
        start_idx = sentence.find(token, current_pos)
        if start_idx == -1:
            raise ValueError(f"Token '{token}' not found in sentence starting from position {current_pos}")
        end_idx = start_idx + len(token)
        offsets.append((start_idx, end_idx))
        current_pos = end_idx
    return offsets

def convert_tokens_to_sentence_and_extract_ne(row):
    tokens = row["tokens"]
    ner_tags_list = row["ner_tags_str"]
    
    # convert tokens to sentence
    sentence = convert_tokens_to_phrase(tokens)
    
    entities = [] # list of dict
    current_entity = []
    current_entity_type = None
    token_offsets = get_token_offsets(sentence, tokens)
    start_pos = None
    end_pos = None
    
    for token, tag, (start, end) in zip(tokens, ner_tags_list, token_offsets):
        if tag == "O": # reset case
            if current_entity:
                entities.append({
                    "type": current_entity_type,
                    "text": convert_tokens_to_phrase(current_entity),
                    "start": start_pos,
                    "end": end_pos
                })
                current_entity = []
                current_entity_type = None
        elif tag.startswith("B-"):
            if current_entity: 
                # store the previous entity before starting a new one
                entities.append({
                    "type": current_entity_type,
                    "text": convert_tokens_to_phrase(current_entity),
                    "start": start_pos,
                    "end": end_pos
                })
            current_entity_type = tag[2:]  # Get the entity type after "B-"
            current_entity = [token]
            start_pos = start
            end_pos = end
        elif tag.startswith("I-") and current_entity_type == tag[2:]:
            # continue the current entity
            current_entity.append(token)
            end_pos = end
        else:
            # handle unexpected cases (e.g., I- tag without a preceding B- tag or mismatched entity types)
            if current_entity:
                entities.append({
                    "type": current_entity_type,
                    "text": convert_tokens_to_phrase(current_entity),
                    "start": start_pos,
                    "end": end_pos
                })
            current_entity = []
            current_entity_type = None
    
    # Check if there's an entity at the end of the sentence
    if current_entity:
        entities.append({
            "type": current_entity_type,
            "text": convert_tokens_to_phrase(current_entity),
            "start": start_pos,
            "end": end_pos
        })
    
    return {"sentence": sentence, "entities": entities}

In [ ]:
ner_ds_extracted = ner_ds_mapped.map(convert_tokens_to_sentence_and_extract_ne)
ner_ds_extracted

In [ ]:
# only keep sentence and entities columns
ner_ds_filtered = ner_ds_extracted.remove_columns([col for col in ner_ds_extracted.column_names if col not in ["tokens", "sentence", "entities"]])
ner_ds_filtered

In [ ]:
# convert dataset to dataframe and serialize the list of dics as JSON strings
# to make sure that there is comma between dicts after saving to csv
df_ner_ds = pd.DataFrame(ner_ds_filtered)
df_ner_ds["entities"] = df_ner_ds["entities"].apply(json.dumps)
df_ner_ds["tokens"] = df_ner_ds["tokens"].apply(json.dumps)
df_ner_ds.head()

In [ ]:
# save the processed dataset to a csv file for evaluation
df_ner_ds.to_csv("eval/ner_detection_eval.csv", index=False)

### Run `mailcom` on the evaluation dataset

In [ ]:
# load workflow configuration
new_settings = {
    "default_lang": "en", # only English in the dataset
    "pseudo_fields": ["content"], # we don't consider subject here
    "pseudo_emailaddresses": False,
    "pseudo_ne": True, # we want to pseudo NE for this evaluation
    "pseudo_numbers": False,
    "datetime_detection": False,
}

# save the updated configuration to a file for reproducibility purposes
new_settings_dir = "./eval"
workflow_settings = mailcom.get_workflow_settings(new_settings=new_settings, 
                                                  updated_setting_dir= new_settings_dir,
                                                  save_updated_settings=True)

In [ ]:
# load csv file into input handler
input_csv = "eval/ner_detection_eval.csv"
# the columns of the csv that should be passed through the processing pipeline/retained in the pipeline
matching_columns = ["sentence"]
# the predefined keys that should be used to match these columns, in the correct order
pre_defined_keys = ["content"]
# what to call any columns that are not matched to pre-defined keys
# get this from the workflow settings
unmatched_keyword = workflow_settings.get("csv_col_unmatched_keyword")

input_handler = mailcom.get_input_handler(in_path=input_csv, in_type="csv", 
                                          col_names=matching_columns, 
                                          init_data_fields=pre_defined_keys, 
                                          unmatched_keyword=unmatched_keyword)

In [ ]:
# process the input data
mailcom.process_data(input_handler.get_email_list(), workflow_settings)
# took around 2 minutes 35.2 seconds on SSC14 (desktop)
# on laptop, 32 GB RAM, no GPU, it took 16 minutes 48.3 seconds (other applications were running too)

In [ ]:
# convert the processed data into a dataframe
ner_df = pd.DataFrame(input_handler.get_email_list())
ner_df.head()

In [ ]:
# only keep the content,  pseudo_content, and sentences columns
filtered_ner_df = ner_df[["content", "pseudo_content", "sentences", "ne_list"]]

In [ ]:
# add pseudo_content to the original dataset
org_ner_df = pd.read_csv(input_csv)
# deserialize JSON back to Python objects for entities and tokens columns
org_ner_df["entities"] = org_ner_df["entities"].apply(json.loads)
org_ner_df["tokens"] = org_ner_df["tokens"].apply(json.loads)

# check before merging
(org_ner_df["sentence"].sort_values().reset_index(drop=True) ==
 filtered_ner_df["content"].sort_values().reset_index(drop=True)).all()

In [ ]:
# merge the pseudo_content into the original dataset
merged_ner_df = org_ner_df.merge(filtered_ner_df, left_on="sentence", right_on="content", how="inner")

assert len(merged_ner_df) == len(org_ner_df) == len(filtered_ner_df)

In [ ]:
merged_ner_df = merged_ner_df.drop(columns=["content"])
merged_ner_df.head()

In [ ]:
# calculate tp, fp, fn for each row
def calculate_tp_fp_fn(row):
    gold_data = row["entities"]
    pred_data = row["ne_list"].get("content", [])

    # normalize data to get tuple of (type, text, start, end)
    norm_gold_data = set(
        (e["type"], e["text"], e["start"], e["end"]) for e in gold_data
    )
    norm_pred_data = set(
        (e["entity_group"], e["word"], e["start"], e["end"]) for e in pred_data
    )

    tp = len(norm_gold_data & norm_pred_data)
    fp = len(norm_pred_data - norm_gold_data)
    fn = len(norm_gold_data - norm_pred_data)

    return pd.Series({"tp": tp, "fp": fp, "fn": fn})

In [ ]:
def cal_eval_metrics(dataframe):
    # add tp, fp, fn columns to the dataframe
    metrics_df = dataframe.apply(calculate_tp_fp_fn, axis=1)
    result_df = pd.concat([dataframe, metrics_df], axis=1)

    # calculate precision, recall, and F1 score for all rows
    tp = result_df["tp"].sum()
    fp = result_df["fp"].sum()
    fn = result_df["fn"].sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1_score:.4f}")

    return result_df


In [ ]:
result_df = cal_eval_metrics(merged_ner_df)
result_df.head()
# Precision: 0.7707
# Recall: 0.8401
# F1 Score: 0.8039

In [ ]:
# save the results to a csv file for further analysis
result_df["entities"] = result_df["entities"].apply(json.dumps)
result_df["ne_list"] = result_df["ne_list"].apply(json.dumps)
result_df["tokens"] = result_df["tokens"].apply(json.dumps)
result_df.to_csv("eval/ner_detection_results.csv", index=False)

In [ ]:
# originally, each content only has one sentence
# check if sentensizing work as expected
merged_ner_df["sentences"].apply(lambda x: len(x.get("content", []))).value_counts()
# sentences -- count
# 1 -- 11531
# 2 -- 60
# 3 -- 6

In [ ]:
# check if non-single sentence cases affect the evaluation results
single_sentence_df = merged_ner_df[merged_ner_df["sentences"].apply(lambda x: len(x.get("content", [])) == 1)]
single_result_df = cal_eval_metrics(single_sentence_df)
single_result_df.head()
# Precision: 0.7743
# Recall: 0.8431
# F1 Score: 0.8072

In [ ]:
# check double sentence cases
double_sentence_df = merged_ner_df[merged_ner_df["sentences"].apply(lambda x: len(x.get("content", [])) == 2)]
double_result_df = cal_eval_metrics(double_sentence_df)
double_result_df.head()
# Precision: 0.3150
# Recall: 0.3883
# F1 Score: 0.3478

In [ ]:
# check triple sentence cases
triple_sentence_df = merged_ner_df[merged_ner_df["sentences"].apply(lambda x: len(x.get("content", [])) == 3)]
triple_result_df = cal_eval_metrics(triple_sentence_df)
triple_result_df.head()
# Precision: 0.0000
# Recall: 0.0000
# F1 Score: 0.0000

### Post analysis

Since Presidio does not consider MISC, recalculate the metrics by excluding MISC from both the ground-truth and the detected entities.

In [ ]:
# load the results saved above
result_df = pd.read_csv("eval/ner_detection_results.csv")
# deserialize JSON back to Python objects for entities and ne_list columns
result_df["entities"] = result_df["entities"].apply(json.loads)
result_df["ne_list"] = result_df["ne_list"].apply(json.loads)
result_df.head()

In [ ]:
considered_types = ["PER", "ORG", "LOC"] # we only consider these 3 types as Presidio does not consider MISC

In [ ]:
# calculate tp, fp, fn for each row
def calculate_tp_fp_fn(row):
    gold_data = row["entities"]
    pred_data = row["ne_list"].get("content", [])

    # normalize data to get tuple of (type, text, start, end)
    norm_gold_data = set(
        (e["type"], e["text"], e["start"], e["end"]) for e in gold_data if e["type"] in considered_types
    )
    norm_pred_data = set(
        (e["entity_group"], e["word"], e["start"], e["end"]) for e in pred_data if e["entity_group"] in considered_types
    )

    tp = len(norm_gold_data & norm_pred_data)
    fp = len(norm_pred_data - norm_gold_data)
    fn = len(norm_gold_data - norm_pred_data)

    return pd.Series({"tp_filtered": tp, "fp_filtered": fp, "fn_filtered": fn})

In [ ]:
def cal_eval_metrics(dataframe):
    # add tp, fp, fn columns to the dataframe
    metrics_df = dataframe.apply(calculate_tp_fp_fn, axis=1)
    result_df = pd.concat([dataframe, metrics_df], axis=1)

    # calculate precision, recall, and F1 score for all rows
    tp = result_df["tp_filtered"].sum()
    fp = result_df["fp_filtered"].sum()
    fn = result_df["fn_filtered"].sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    print(f"Precision - filtered: {precision:.4f}")
    print(f"Recall -filtered: {recall:.4f}")
    print(f"F1 Score - filtered: {f1_score:.4f}")

    return result_df

In [ ]:
post_result_df = cal_eval_metrics(result_df)
post_result_df.head()
# Precision - filtered: 0.8825
# Recall -filtered: 0.8760
# F1 Score - filtered: 0.8792

## Numerical data detection

No suitable dataset was found so far...